In [124]:
# Strategy
## 1) Vector store created with RecursiveCharacterTextSplitter (chunking)
## and HuggingFaceEmbeddings
## 2) Query vector store for a test case using retriever object
## 3)  

In [125]:
import os
import json
import glob
import math

from dotenv import load_dotenv
from pathlib import Path
from typing import List
from openai import OpenAI
from pydantic import BaseModel, Field

In [126]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_core.messages import SystemMessage, HumanMessage

### Vectorstore Creation

In [127]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

In [128]:
RETRIEVAL_K = 20
CHUNK_SIZE = 1000

In [129]:
load_dotenv(override=True)

True

In [130]:
def normalize_section(text):
    # Remove extra spaces around hyphens and colons
    return re.sub(r'\s*[-:]\s*', lambda m: m.group().strip(), text).strip()

In [131]:
def load_json_with_root(filepath):
    with open(filepath, 'r') as f:
        full_data = json.load(f)
        policy_name = full_data.get("policy_name", "unknown")
        category = full_data.get("category", "unknown")
        source = full_data.get("source_path", "unknown").split('\\')[1]
        
    def metadata_func(record: dict, base_metadata: dict):
        base_metadata['policy_name'] = policy_name
        base_metadata['category'] = category
        base_metadata['source'] = source
        base_metadata['page_type'] = record.get("page_type", "unknown")
        return base_metadata
    
    return JSONLoader(
        file_path=filepath,
        jq_schema='.pages[] | select(.page_type == "content")',
        content_key='text',
        metadata_func=metadata_func
    )

In [132]:
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    loader = DirectoryLoader(folder, glob='**/*.json', loader_cls=load_json_with_root)
    folder_docs = loader.load()
    
    for doc in folder_docs:
        documents.append(doc)
        
print(len(documents))

432


#### Text Splitters

In [133]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
len(chunks)

1922

In [134]:
# hf_embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [135]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()
    
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)

### Test Case Generation

In [136]:
class TestQuestion(BaseModel):
    test_id: str = Field(description="Unique identifier for the test case")
    query: str = Field(description="The user question to be asked to the RAG system")
    source_doc: str = Field(description="The policy document(s) where the answer should come from")
    expected_answer: str = Field(description="The correct ground truth answer for evaluation")
    relevant_sections: list = Field(description="The sections in the document where the answer lives")
    question_type: str = Field(description="Category of the question")
    difficulty: str = Field(description="Complexity level of the question")

In [137]:
def load_tests() -> List[TestQuestion]:
    tests = []
    with open("tests.jsonl", 'r', encoding='utf-8') as f:
        for line in f:
            test = json.loads(line.strip())
            tests.append(TestQuestion(**test))
            
    return tests

In [138]:
tests = load_tests()
len(tests)

60

### LLM Answers

In [139]:
policies = []

loc = "knowledge-base"
dirs = os.listdir(loc)

for d in dirs:
    path = Path(f"{loc}/{d}")
    for file in path.iterdir():
        if file.is_file:
            policy = file.name.split('.')[0].lower()
            policies.append(policy)
#             print(policy[:45], len(policy))

In [140]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [141]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the LIC (Life Insurance Corporation of India).
You are chatting with a user about LIC's insurance products only.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [142]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [143]:
answer_question("What are the two death benefit options available under LIC Digi Term?", [])

'The LIC Digi Term plan offers two death benefit options to choose from:\n\n1. Lump Sum Death Benefit: The entire sum assured is paid to the nominee or legal heir in a lump sum amount.\n2. Income Option: The death benefit is paid in the form of regular income installments over a specified period.\n\nYou can select either of these options at the time of policy purchase or as per your preference. If you need more details or assistance in choosing the right option, please let me know!'

### Retrieval Evaluation

In [144]:
example = tests[0]
example

TestQuestion(test_id='TC_001', query='What is the death benefit payable under LIC Jeevan Tarun?', source_doc='jeevan_tarun.json', expected_answer='The death benefit under Jeevan Tarun is the Sum Assured on Death along with vested Simple Reversionary Bonuses and Final Additional Bonus, if any. The Sum Assured on Death is defined as the higher of 125% of Basic Sum Assured or 7 times of Annualized Premium. This benefit shall not be less than 105% of the total premiums paid up to the date of death.', relevant_sections=['PART-C: BENEFITS'], question_type='factual', difficulty='easy')

In [145]:
import re

In [146]:
def normalize_section(text):
    # Remove extra spaces around hyphens and colons
    return re.sub(r'\s*[-:]\s*', lambda m: m.group().strip(), text).strip()

In [147]:
# # retriever = vectorstore.as_retriever()
# kwargs = {
#     "search_kwargs": {
#         "filter": {
#             "policy_name": next((p for p in policies if p in example.query.lower()), None)
#         }
#     }
# }
# documents = retriever.invoke(example.query, config=kwargs)

In [148]:
def calculate_mrr(docs, section):
    for rank, doc in enumerate(docs, start=1):
        relevant_section = normalize_section(section)
        if relevant_section.lower() in doc.page_content.lower(): #page_content is already normalized
            return 1.0 / rank
    return 0.0

In [149]:
# def calculate_dcg(docs, question, k):
#     dcg = 0
#     relevant_section = normalize_section(question.relevant_section)
#     relevences = [1 if relevant_section in doc.page_content else 0 for doc in docs]
    
#     for i in range(min(k, len(relevences))):
#         dcg += relevences[i] / math.log2(i+2)
#     return dcg

In [150]:
def calculate_ndcg(docs, question, k):
    relevant_section = normalize_section(question.relevant_section)
    relevances = [1 if relevant_section in doc.page_content else 0 for doc in docs]
    
    # DCG
    dcg = sum(relevances[i] / math.log2(i + 2) for i in range(min(k, len(relevances))))
    
    # Ideal DCG — best possible ranking
    ideal_relevances = sorted(relevances, reverse=True)
    idcg = sum(ideal_relevances[i] / math.log2(i + 2) for i in range(min(k, len(ideal_relevances))))
    
    return dcg / idcg if idcg > 0 else 0.0

In [151]:
def calculate_hit_rate(docs, question):
    relevant_section = normalize_section(question.relevant_section)
    tot_relevant_docs = sum(1 if relevant_section in doc.page_content else 0 for doc in docs)
    return tot_relevant_docs / len(docs)

In [152]:
# def calculate_recall_k(docs, question, total_relevant, top_k=3):
#     relevant_section = normalize_section(question.relevant_section)
#     top_docs = sum(
#         1 for doc in docs[:top_k] 
#         if relevant_section in doc.page_content
#     )
#     return top_docs / total_relevant if total_relevant > 0 else 0.0

In [153]:
# calculate_mrr(documents, example)

In [154]:
# calculate_dcg(chunks, example, TOP_K)

In [155]:
# calculate_ndcg(documents, example, TOP_K)

In [156]:
# calculate_hit_rate(documents, example)

In [157]:
# calculate_recall_k(documents, example, chunks)

In [158]:
# Single test case
# mrr=0.25, ndcg=0.5, hitrate=40%

In [159]:
# Complexity => source_doc = "multiple", relevant_section = "multiple"

In [160]:
def get_kwargs(question):
    return {
    "search_kwargs": {
        "filter": {
            "policy_name": next((p for p in policies if p in question.lower()), None)
        }
    }
}

In [161]:
def evaluate_test(test):
    kwargs = get_kwargs(test.query)
    docs = retriever.invoke(test.query, config=kwargs, k=RETRIEVAL_K)
    mrr = [calculate_mrr(docs, section) for section in test.relevant_sections]
    avg_mrr = sum(mrr) / len(mrr) if mrr else 0.0
    return avg_mrr

In [162]:
def evaluate_all_tests(tests):
    for test in tests:
        result = evaluate_test(test)
        yield result

In [163]:
#evaluate_all_tests(tests)

In [164]:
tot_mrr = 0
for result in evaluate_all_tests(tests):
    tot_mrr += result
    
print(tot_mrr / 60)

0.045662277537277536
